# Self-Supervised Fisheye Rectification — the repo's Quick Start on Kaggle (NYU Depth V2)

This notebook runs the steps from the repo's README "Quick start":

```
1. git clone https://github.com/memara111/SelfSupervisedFisheyeRectification.git
2. pip3 install -r requirements.txt
3. python3 tools/prepare_cityscapes.py     -> replaced by NYU Depth V2 (see step 3)
4. python3 src/train.py cfg/cityscapes.yml -> cfg/nyu2.yml
5. python3 src/test.py  cfg/cityscapes.yml -> cfg/nyu2.yml
```

What differs from the README, and why:

| step | change |
|---|---|
| 1 | the clone is pinned to `REPO_REF`, **the branch that contains the pipeline fixes**; `main` still has the pre-fix code |
| 2 | Kaggle already ships a CUDA build of torch/torchvision, so only *missing* packages are installed - a blind `pip install -r requirements.txt` can replace the GPU build with a multi-GB PyPI wheel |
| 3 | Cityscapes is a 2 GB download with an account login; NYU Depth V2 is already attached here. `data/nyu2/{train,val,test}.lst` are written in exactly the format `tools/prepare_cityscapes.py` produces |
| 4, 5 | same commands, plus `--resume` for training |

### Why the branch matters

In the pre-fix code the loss was applied on the **wrong side** of the geometric model: it re-used
the same forward render map that produced the labels, so minimising it produced a *double*
distortion and its optimum sat at a parameter of the wrong sign (measured: `-0.295` for a ground
truth of `+0.9`). A run on that code still looks healthy - the loss goes down - but what it
converges to is not the distortion. `test.py` / `predict.py` re-applied the render as well, so the
"rectified" image was a second distortion rather than an undoing of the first, and the output of
the network was unbounded (`1 - |d|^2 < 0` -> NaN loss) which the quadratic form also hit exactly
at `k = 0`, i.e. at the first fragment of every curriculum stage.

The fixed branch rectifies with the exact inverse of the render map (`src/core/division_model.py`,
shared by the loss, the renderer and both scripts), uses all three key-point coordinates, bounds
the head to the parameterisation, and ships `tests/run_tests.py` (31 tests: the involution, the
argmin of the loss, per-axis frame bounds, and a train -> test -> predict -> resume run).

### Resuming

`src/train.py` resumes only when given **`--resume`** - a fresh run no longer silently inherits a
checkpoint that happens to be lying around. The checkpoint (model, optimizer, loss history, and
the distortion range it was trained with) is written after *every* epoch, before plotting, so
Kaggle's session limit costs at most the epoch in progress.

## 1. Clone the project

Pinned to the branch with the fixes (`REPO_REF`); switch it to `"main"` once they are merged.

In [ ]:
import os

REPO_URL = "https://github.com/memara111/SelfSupervisedFisheyeRectification.git"
# The pipeline fixes live on this branch; once they are merged into main, set REPO_REF = "main".
REPO_REF = "arena/01a063d1-selfsupervisedfisheyerectifica"
REPO_DIR = "/kaggle/working/SelfSupervisedFisheyeRectification"

os.chdir("/kaggle/working")
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(REPO_DIR + " already exists -> updating to " + REPO_REF)
    !git -C {REPO_DIR} fetch --depth 1 origin {REPO_REF}
    !git -C {REPO_DIR} reset --hard FETCH_HEAD
else:
    !git clone --depth 1 --branch {REPO_REF} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!pwd
!git log --oneline -1

## 2. Install the dependencies

In [ ]:
# requirements.txt pins the real dependency set, but Kaggle's image already ships a CUDA build of
# torch / torchvision. Reinstalling those would be a multi-GB download that can swap the GPU build
# for a CPU wheel, so install only what is genuinely missing and no more.
import importlib

MODULES = [
    ("yaml", "pyyaml"),
    ("numpy", "numpy"),
    ("PIL", "pillow"),
    ("matplotlib", "matplotlib"),
    ("torch", "torch"),
    ("torchvision", "torchvision"),
    ("torchsummary", "torchsummary"),   # src/train.py imports it for the layer summary
]

missing = []
for module, pip_name in MODULES:
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(pip_name)

if not os.path.isfile("src/core/division_model.py"):
    # a pre-fix clone still imports scipy (its sparse resampler) and cv2, which the fixed
    # code dropped along with the sparse matrix - they are only needed on that branch
    for module, pip_name in (("scipy", "scipy"), ("cv2", "opencv-python-headless")):
        try:
            importlib.import_module(module)
        except ImportError:
            missing.append(pip_name)

print("requirements.txt pins:", " ".join(open("requirements.txt").read().split()))
print("missing in this image:", " ".join(missing) or "nothing")

if missing:
    TO_INSTALL = " ".join(missing)
    !pip3 install {TO_INSTALL} --break-system-packages -q
    importlib.invalidate_caches()
    for module, _ in MODULES:
        importlib.import_module(module)   # fail here rather than three cells later
print("all imports OK")

### 2.5 Compatibility patches (only needed for a *pre-fix* clone)

Each patch below is applied only if the file still contains the code it targets, so on the fix
branch this cell does nothing at all and is safe to re-run. What it covers:

- `src/core/config.py` - `yaml.load(f)` without a `Loader=` is a hard `TypeError` since PyYAML 5.1.
- `src/models/pem.py` - `torch.hub.load(...)` needs GitHub access just to *construct* the model.
- `src/train.py` - calls `plt.plot(...)` without importing `plt`, which aborted the run right after
  the first epoch (after training, before the checkpoint was saved).
- `torch.load` calls - explicit `map_location` / `weights_only`, which newer PyTorch changed.

**Read this before using the patches:** they make the old scripts *run*, they do not make them
*correct*. Two things the old renderer costs you even after patching: it builds a sparse
`(3HW x 3HW)` matrix per effector (measured: OOM-killed at 192x256 on a 4 GB machine, where the
fixed gather-based resampler needs ~8 MB), and it truncates sample coordinates to integers instead
of interpolating, which aliases out-of-frame pixels onto valid ones. Neither is fixable from a
notebook cell - they are why this notebook pins the clone to `REPO_REF`.

In [ ]:
import os

# Every group is applied only if the guard text is still present, i.e. only for a clone that
# predates the fix. On the fix branch all four groups report "already fixed" and nothing is written.
def patch_file(path, groups):
    if not os.path.exists(path):
        print("  skip   {} (no such file)".format(path))
        return False
    with open(path, "r") as f:
        content = f.read()
    untouched = content
    for label, guard, edits in groups:
        if guard not in content:
            print("  ok     {} - {}".format(path, label))
            continue
        for old, new in edits:
            content = content.replace(old, new)
        print("  PATCH  {} - {}".format(path, label))
    if content != untouched:
        with open(path, "w") as f:
            f.write(content)
    return content != untouched


patch_file("src/core/config.py", [
    ("yaml.load(f) needs a Loader on PyYAML >= 5.1", "self.config = yaml.load(f)",
     [("self.config = yaml.load(f)", "self.config = yaml.safe_load(f)")]),
])

patch_file("src/models/pem.py", [
    ("build the backbone without a torch.hub round trip",
     "torch.hub.load('pytorch/vision:v0.6.0', 'vgg11', pretrained=False)",
     [("import torchvision.transforms as transforms",
       "import torchvision.transforms as transforms\nimport torchvision.models as tvmodels"),
      ("torch.hub.load('pytorch/vision:v0.6.0', 'vgg11', pretrained=False)",
       "tvmodels.vgg11(weights=None)")]),
])

patch_file("src/train.py", [
    ("plt is used but never imported", "import argparse\nimport os\nimport torch",
     [("import argparse\nimport os\nimport torch",
       "import argparse\nimport os\nimport torch\nimport matplotlib\nmatplotlib.use('Agg')\nimport matplotlib.pyplot as plt")]),
    ("torch.load on a checkpoint that also holds the optimizer state",
     "checkpoint = torch.load(model_state_file)",
     [("checkpoint = torch.load(model_state_file)",
       "checkpoint = torch.load(model_state_file, map_location=DEVICE, weights_only=False)")]),
])

for script in ("src/test.py", "src/predict.py"):
    patch_file(script, [
        ("torch.load needs an explicit map_location/weights_only", "checkpoint = torch.load(model_file)",
         [("checkpoint = torch.load(model_file)",
           "checkpoint = torch.load(model_file, map_location=DEVICE, weights_only=False)")]),
    ])

### 2.6 Which clone am I actually running?

The repo ships 31 dependency-free regression tests (geometry, the argmin of the loss, the
renderer, and a full train -> test -> predict -> resume run) - about a minute on CPU. Uncomment
the last line if you want to watch them.

In [ ]:
import os
import subprocess

head = subprocess.run(["git", "log", "--oneline", "-1"], stdout=subprocess.PIPE,
                      text=True).stdout.strip()
print("clone HEAD :", head or "(not a git checkout)")
if os.path.isfile("src/core/division_model.py"):
    print("fix status : FIXED - src/core/division_model.py + tests/run_tests.py are present")
else:
    print("fix status : PRE-FIX. The patches above make these scripts *run*; the loss is still "
          "the double-distortion one, so do not read anything into the curves. Set REPO_REF in "
          "cell 2 to the fix branch for correct behaviour.")

# optional, ~1 minute on CPU:
# !python3 tests/run_tests.py

## 3. Prepare the dataset (NYU Depth V2 instead of Cityscapes)

Builds `data/nyu2/train.lst`, `val.lst` and `test.lst` - one absolute image path per line, which is
the format `DistortDataset` expects and the same one `tools/prepare_cityscapes.py` writes. Only the
`.jpg` RGB frames are used; the `.png` depth maps are not needed for a self-supervised task. The
split is done **by scene folder**, so validation and test scenes are fully held out.

Training images are capped at `MAX_TRAIN_IMAGES`. The validation set is deliberately small: it is
re-rendered and evaluated after *every* epoch, so an unconstrained split costs more per epoch than
training does.

In [ ]:
import glob
import os
import random

# ----------------------------- EDIT THESE -----------------------------------
DATASET_ROOT = "/kaggle/input/datasets/soumikrakshit/nyu-depth-v2/nyu_data/data/nyu2_train"
MAX_TRAIN_IMAGES = 1000     # hard cap on training images (None = no cap)
MAX_VAL_IMAGES = 150        # evaluated after EVERY epoch -> keep it small
MAX_TEST_IMAGES = 150       # only used by src/test.py
VAL_SPLIT = 0.05            # fraction of *scenes* held out
TEST_SPLIT = 0.05
SEED = 42
# ------------------------------------------------------------------------------

random.seed(SEED)

if not os.path.isdir(DATASET_ROOT):
    raise FileNotFoundError(
        "Couldn't find {}. Attach the NYU Depth V2 dataset to this notebook (Add Data) and "
        "adjust DATASET_ROOT.".format(DATASET_ROOT))

scene_dirs = sorted(d for d in glob.glob(os.path.join(DATASET_ROOT, "*")) if os.path.isdir(d))
random.shuffle(scene_dirs)
n_val = max(1, int(len(scene_dirs) * VAL_SPLIT))
n_test = max(1, int(len(scene_dirs) * TEST_SPLIT))
val_scenes, test_scenes, train_scenes = scene_dirs[:n_val], scene_dirs[n_val:n_val + n_test], \
    scene_dirs[n_val + n_test:]
print("{} scenes -> {}/{} held out for val/test".format(
    len(scene_dirs), len(val_scenes), len(test_scenes)))


def collect(scenes, limit=None):
    # split by scene, so a validation scene never shares a folder with a training one
    images = []
    for scene in scenes:
        images.extend(sorted(glob.glob(os.path.join(scene, "*.jpg"))))
        if limit and len(images) >= limit:
            break
    if limit:
        images = images[:limit]
    return images


train_images = collect(train_scenes)
if MAX_TRAIN_IMAGES and len(train_images) > MAX_TRAIN_IMAGES:
    random.shuffle(train_images)
    train_images = train_images[:MAX_TRAIN_IMAGES]
val_images = collect(val_scenes, MAX_VAL_IMAGES)
test_images = collect(test_scenes, MAX_TEST_IMAGES)
print("{} train / {} val / {} test images".format(
    len(train_images), len(val_images), len(test_images)))
assert train_images and val_images and test_images, "no .jpg images found - check DATASET_ROOT"

# the scripts resolve the split lists as <data_root>/<DATASET.NAME>/<split>, and --data_root
# defaults to "data", i.e. data/nyu2 for cfg/nyu2.yml
data_path = os.path.abspath("data/nyu2")
os.makedirs(data_path, exist_ok=True)
for name, images in (("train", train_images), ("val", val_images), ("test", test_images)):
    with open(os.path.join(data_path, name + ".lst"), "w") as f:
        f.write("".join(image + "\n" for image in images))
print("wrote", data_path)
print("example line:", train_images[0])

### Write `cfg/nyu2.yml`

Same structure as `cfg/cityscapes.yml`. Every key below has a default in the scripts, so this file
is mostly documentation of what is being used - except `DATASET.NAME`, `HEIGHT`/`WIDTH`, which are
required.

In [ ]:
nyu2_yml = '''DATASET:
  NAME: nyu2
  # NYU frames are 640x480: keep that ratio. (144x256 stretched every image 27% wider
  # before the renderer ever saw it, which biases the geometry it is asked to invert.)
  HEIGHT: 192
  WIDTH: 256
  CROP: true

MODEL:
  MAX_DISTORTION: 1.0      # bound of the network output. Recorded in the checkpoint;
                           # test.py / predict.py refuse a config that disagrees with it.
  VGG_PRETRAINED: false    # true = ImageNet weights via torchvision (needs internet).
                           # Worth trying first if the loss stalls on ~1k images.
  FREEZE_VGG: false

TRAIN:
  MAX_EPOCH: 1000
  BATCH_SIZE: 16
  LEARNING_RATE: 0.0001
  NUM_WORKERS: 2
  LOG_INTERVAL: 20
  CURRICULUM:
    ENABLED: true
    SWITCH_EPOCH: 30
    MAX_DISTORTION: 0.9    # the range the data is RENDERED with. Must stay strictly
                           # inside MODEL.MAX_DISTORTION: a squashed output can only
                           # approach its own bound asymptotically.

TEST:
  CHECKPOINT: checkpoint.pth.tar
  SAVE_RESULTS: true
  OUTPUT_DIR: results
  NUM_FRAGMENTS: 10
  MAX_DISTORTION: 0.9
  OUTPUT_SIZE:             # size of the saved _rec.jpg. It may differ from the training
    HEIGHT: 384            # size (test.py resamples first); on the pre-fix code a
    WIDTH: 512             # different value silently letterboxed the result in black

'''

os.makedirs("cfg", exist_ok=True)
with open("cfg/nyu2.yml", "w") as f:
    f.write(nyu2_yml)
print(nyu2_yml)

## 4. Training

```
python3 src/train.py cfg/nyu2.yml --resume
```

`MAX_EPOCH: 1000` is long by design (it matches the repo's config); Kaggle's session limit will cut
it off. Re-running this cell continues from the last completed epoch, and to carry a run into a
brand-new session, download `outputs/nyu2/checkpoint.pth.tar`, attach it as an input dataset and
uncomment the copy below.

The metric to watch is not the loss alone: with the fixed objective a low loss means the predicted
`k` reproduces the key-point relations. `src/test.py` (step 6) is what turns that into the number
you care about - the mean absolute error on `k`, plus the saved rectifications.

In [ ]:
# Continuing an interrupted run. To resume in a brand-new Kaggle session, attach the previous
# run's outputs as an input dataset and uncomment:
#
#   import shutil
#   os.makedirs("outputs/nyu2", exist_ok=True)
#   shutil.copy("/kaggle/input/<your-dataset>/checkpoint.pth.tar",
#               "outputs/nyu2/checkpoint.pth.tar")

!python3 src/train.py cfg/nyu2.yml --resume

## 5. Plot the train / val loss

`src/train.py` writes `outputs/nyu2/losses.png` *and* `outputs/nyu2/losses.json` after every epoch.

In [ ]:
import json
import os

import matplotlib.pyplot as plt
from PIL import Image

losses_png = "outputs/nyu2/losses.png"
if not os.path.isfile(losses_png):
    print("no {} yet - run the training cell first".format(losses_png))
else:
    plt.figure(figsize=(8, 5))
    plt.imshow(Image.open(losses_png))
    plt.axis("off")
    plt.show()
    with open("outputs/nyu2/losses.json") as f:
        history = json.load(f)
    print("epochs: {} | last train {:.6f} | last val {:.6f}".format(
        len(history["train"]), history["train"][-1], history["val"][-1]))
    print("also at", os.path.abspath(losses_png))

### Optional: re-plot from the checkpoint

Same history as raw numbers, straight from the checkpoint.

In [ ]:
import matplotlib.pyplot as plt
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# the fixed checkpoint stores only tensors + plain python types, so the safe loader is enough
ckpt = torch.load("outputs/nyu2/checkpoint.pth.tar", map_location=DEVICE, weights_only=True)
train_losses, val_losses = ckpt["train_losses"], ckpt["val_losses"]

plt.figure(figsize=(7, 5))
plt.plot(range(1, len(train_losses) + 1), train_losses, label="train")
plt.plot(range(1, len(val_losses) + 1), val_losses, label="val")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.yscale("log")
plt.title("train / val loss (epoch {})".format(ckpt["epoch"]))
plt.legend()
plt.grid(alpha=0.3)
plt.show()

for key in ("epoch", "model_max_distortion", "data_max_distortion"):
    if key in ckpt:
        print("{} = {}".format(key, ckpt[key]))

del ckpt  # a checkpoint with optimizer states is ~1.5 GB; a notebook kernel would keep it alive

## 6. Testing

```
python3 src/test.py cfg/nyu2.yml
```

Renders each image from `data/nyu2/test.lst` with a known `k` over a symmetric sweep
(`TEST.MAX_DISTORTION`), predicts it, and - because `TEST.SAVE_RESULTS: true` - writes
`{idx}_org.jpg` (clean input), `{idx}_dis.jpg` (what the network sees), `{idx}_rec.jpg`
(the model's rectification at `TEST.OUTPUT_SIZE`) plus `metrics.csv`
(`idx,path,gt,prediction,abs_err,relation_err_pred,relation_err_gt`) into `results/`.

In [ ]:
!python3 src/test.py cfg/nyu2.yml

### The saved triplets, annotated with `gt` / `prediction` from `metrics.csv`

In [ ]:
import csv
import glob
import os

import matplotlib.pyplot as plt
from PIL import Image

RESULT_DIR = "results"        # TEST.OUTPUT_DIR
SUFFIX = "_org.jpg"


def stem(path):
    # src/test.py saves {index}_org.jpg | _dis.jpg | _rec.jpg
    return os.path.basename(path)[: -len(SUFFIX)]


def sort_key(path):
    name = stem(path)
    return (0, int(name), "") if name.isdigit() else (1, 0, name)


org_files = sorted(glob.glob(os.path.join(RESULT_DIR, "*" + SUFFIX)), key=sort_key)
assert org_files, ("nothing in {}/ - run the testing cell first (it needs TEST.SAVE_RESULTS: "
                   "true)".format(RESULT_DIR))

metrics = {}
metrics_csv = os.path.join(RESULT_DIR, "metrics.csv")
if os.path.isfile(metrics_csv):
    with open(metrics_csv, newline="") as f:
        metrics = {row["idx"]: row for row in csv.DictReader(f)}

panels = [("_org.jpg", "original (clean)"), ("_dis.jpg", "distorted (model input)"),
          ("_rec.jpg", "rectified (model output)")]
show = org_files[:4]
fig, axes = plt.subplots(len(show), 3, figsize=(11, 3.2 * len(show)), squeeze=False)
for row, path in zip(axes, show):
    name = stem(path)
    info = metrics.get(name, {})
    for ax, (suffix, title) in zip(row, panels):
        image_path = os.path.join(RESULT_DIR, name + suffix)
        if os.path.isfile(image_path):
            ax.imshow(Image.open(image_path))
        ax.set_title(title if ax is row[0] else "")
        ax.axis("off")
    if info:
        row[0].set_title("original\ngt {} / pred {} / err {}".format(
            info["gt"], info["prediction"], info["abs_err"]), fontsize=8)

if metrics:
    errors = [float(row["abs_err"]) for row in metrics.values()]
    print("images {} | mean |k - gt| {:.4f} | max {:.4f}".format(
        len(errors), sum(errors) / len(errors), max(errors)))
plt.tight_layout()
plt.show()

## 7. Bonus: rectify your own fisheye photo

`test.py` only sees synthetically distorted NYU frames. `src/predict.py` (not part of the README's
Quick start) takes a directory of real images and needs no ground truth. Note what the model can
and cannot do: it is trained on *synthetic* fisheye renders of rectilinear photos, so a genuine
wide-angle lens is out of distribution - expect a small `|k|` and treat the output as a suggestion.

The fixed `predict.py` (i) rectifies rather than re-distorting, (ii) resizes to the `-ow`/`-oh`
frame you ask for, (iii) refuses a checkpoint whose recorded `MODEL.MAX_DISTORTION` does not match
the config, and (iv) writes `distortions.txt` with `--save-distortions`.

In [ ]:
import os
import random
import shutil

from PIL import Image

# ----------------------------- EDIT THIS -------------------------------------
MY_IMAGE_PATH = "/kaggle/input/datasets/qweasd123456785432/im-calib/im_calib/im_calib_01.jpg"
TARGET_WIDTH = 1280        # the rectified file is rendered at this width
# ------------------------------------------------------------------------------

if not os.path.isfile(MY_IMAGE_PATH):
    # the optional dataset above is not attached: fall back to one of the NYU frames, i.e.
    # exactly the kind of image this model is trained on
    try:
        MY_IMAGE_PATH = random.choice(train_images)
    except NameError:
        raise FileNotFoundError(
            "neither the im_calib image nor data/nyu2 exist - set MY_IMAGE_PATH, or run the "
            "dataset cell first")
    print("MY_IMAGE_PATH not found, using", MY_IMAGE_PATH)

infer_input_dir = "/kaggle/working/my_photo"
# -o / --save-distortions were added by the pipeline fix (a pre-fix predict.py always wrote
# to <input>/out and rejects unknown flags), so only pass them when they exist
fixed_cli = os.path.isfile("src/core/division_model.py")
rectified_dir = os.path.join(infer_input_dir, "rectified" if fixed_cli else "out")
os.makedirs(infer_input_dir, exist_ok=True)
shutil.copy(MY_IMAGE_PATH, os.path.join(infer_input_dir, os.path.basename(MY_IMAGE_PATH)))

# -ow / -oh are the output frame, so pick them from the image instead of forcing 16:9 on it
width, height = Image.open(MY_IMAGE_PATH).size
out_width = TARGET_WIDTH
out_height = max(2, round(TARGET_WIDTH * height / width / 2) * 2)
print("input {}x{} -> rectifying at {}x{}".format(width, height, out_width, out_height))

PREDICT_FLAGS = ("-o {} --save-distortions".format(rectified_dir)) if fixed_cli else ""
!python3 src/predict.py cfg/nyu2.yml -i {infer_input_dir} {PREDICT_FLAGS} -ow {out_width} -oh {out_height}

In [ ]:
import os

import matplotlib.pyplot as plt
from PIL import Image

filename = os.path.basename(MY_IMAGE_PATH)
original_path = os.path.join(infer_input_dir, filename)
candidates = [os.path.join(infer_input_dir, sub, filename) for sub in ("rectified", "out")]
rectified_path = next((p for p in candidates if os.path.isfile(p)), None)
assert rectified_path is not None, \
    "predict.py wrote none of {} - see the output of the cell above".format(candidates)

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
for ax, path, title in zip(axes, (original_path, rectified_path),
                           ("input (as given)", "rectified (as saved)")):
    ax.imshow(Image.open(path))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

distortion_log = os.path.join(os.path.dirname(rectified_path), "distortions.txt")
if os.path.isfile(distortion_log):
    print("predicted division-model parameter (value<TAB>file):")
    print(open(distortion_log).read().strip())
print("saved to", os.path.abspath(rectified_path))

## 8. Notes

- **`outputs/nyu2/` and `results/` are git-ignored** in the repo, so nothing is committed by running
  this notebook; download them from the Kaggle output panel to keep a run.
- **Re-running cells 8-10** regenerates the splits and the config; the checkpoint is not deleted, so
  `--resume` continues from it. Delete `outputs/nyu2/checkpoint.pth.tar` to start over.
- **Dataset**: the `.lst` files hold absolute paths into `/kaggle/input/...`, so a *new* session needs
  cell 8 re-run (the dataset attachment path can change between Kaggle versions).
- Nothing in this notebook changes the repo's tracked files; cell 2.5 does edit the *clone* in
  `/kaggle/working`, which is not pushed anywhere.